In [ ]:
import numpy as np

**Module** is an abstract class which defines fundamental methods necessary for a training a neural network. You do not need to change anything here, just read the comments.

In [ ]:
class Module(object):
    """
    Basically, you can think of a module as of a something (black box)
    which can process `input` data and produce `ouput` data.
    This is like applying a function which is called `forward`:

        output = module.forward(input)

    The module should be able to perform a backward pass: to differentiate the `forward` function.
    More, it should be able to differentiate it if is a part of chain (chain rule).
    The latter implies there is a gradient from previous step of a chain rule.

        gradInput = module.backward(input, gradOutput)
    """
    def __init__ (self):
        self.output = None
        self.gradInput = None
        self.training = True

    def forward(self, input):
        """
        Takes an input object, and computes the corresponding output of the module.
        """
        return self.updateOutput(input)

    def backward(self,input, gradOutput):
        """
        Performs a backpropagation step through the module, with respect to the given input.

        This includes
         - computing a gradient w.r.t. `input` (is needed for further backprop),
         - computing a gradient w.r.t. parameters (to update parameters while optimizing).
        """
        self.updateGradInput(input, gradOutput)
        self.accGradParameters(input, gradOutput)
        return self.gradInput


    def updateOutput(self, input):
        """
        Computes the output using the current parameter set of the class and input.
        This function returns the result which is stored in the `output` field.

        Make sure to both store the data in `output` field and return it.
        """

        # The easiest case:

        # self.output = input
        # return self.output

        pass

    def updateGradInput(self, input, gradOutput):
        """
        Computing the gradient of the module with respect to its own input.
        This is returned in `gradInput`. Also, the `gradInput` state variable is updated accordingly.

        The shape of `gradInput` is always the same as the shape of `input`.

        Make sure to both store the gradients in `gradInput` field and return it.
        """

        # The easiest case:

        # self.gradInput = gradOutput
        # return self.gradInput

        pass

    def accGradParameters(self, input, gradOutput):
        """
        Computing the gradient of the module with respect to its own parameters.
        No need to override if module has no parameters (e.g. ReLU).
        """
        pass

    def zeroGradParameters(self):
        """
        Zeroes `gradParams` variable if the module has params.
        """
        pass

    def getParameters(self):
        """
        Returns a list with its parameters.
        If the module does not have parameters return empty list.
        """
        return []

    def getGradParameters(self):
        """
        Returns a list with gradients with respect to its parameters.
        If the module does not have parameters return empty list.
        """
        return []

    def train(self):
        """
        Sets training mode for the module.
        Training and testing behaviour differs for Dropout, BatchNorm.
        """
        self.training = True

    def evaluate(self):
        """
        Sets evaluation mode for the module.
        Training and testing behaviour differs for Dropout, BatchNorm.
        """
        self.training = False

    def __repr__(self):
        """
        Pretty printing. Should be overrided in every module if you want
        to have readable description.
        """
        return "Module"

# Sequential container

**Define** a forward and backward pass procedures.

In [ ]:
class Sequential(Module):
    """
         This class implements a container, which processes `input` data sequentially.

         `input` is processed by each module (layer) in self.modules consecutively.
         The resulting array is called `output`.
    """

    def __init__ (self):
        super(Sequential, self).__init__()
        self.modules = []

    def add(self, module):
        """
        Adds a module to the container.
        """
        self.modules.append(module)

    def updateOutput(self, input):
        self._forward_inputs = [input]
        out = input
        for module in self.modules:
            out = module.forward(out)
            self._forward_inputs.append(out)
        self.output = out
        return self.output

    def backward(self, input, gradOutput):
        grad = gradOutput
        for i in range(len(self.modules) - 1, -1, -1):
            grad = self.modules[i].backward(self._forward_inputs[i], grad)
        self.gradInput = grad
        return self.gradInput

    def zeroGradParameters(self):
        for module in self.modules:
            module.zeroGradParameters()

    def getParameters(self):
        return [x.getParameters() for x in self.modules]

    def getGradParameters(self):
        return [x.getGradParameters() for x in self.modules]

    def __repr__(self):
        string = "".join([str(x) + '\n' for x in self.modules])
        return string

    def __getitem__(self,x):
        return self.modules.__getitem__(x)

    def train(self):
        self.training = True
        for module in self.modules:
            module.train()

    def evaluate(self):
        self.training = False
        for module in self.modules:
            module.evaluate()


# Layers

## 1 (0.2). Linear transform layer
Also known as dense layer, fully-connected layer, FC-layer, InnerProductLayer (in caffe), affine transform
- input:   **`batch_size x n_feats1`**
- output: **`batch_size x n_feats2`**

In [ ]:
class Linear(Module):
    """
    A module which applies a linear transformation
    A common name is fully-connected layer, InnerProductLayer in caffe.

    The module should work with 2D input of shape (n_samples, n_feature).
    """
    def __init__(self, n_in, n_out):
        super(Linear, self).__init__()

        stdv = 1./np.sqrt(n_in)
        self.W = np.random.uniform(-stdv, stdv, size = (n_out, n_in))
        self.b = np.random.uniform(-stdv, stdv, size = n_out)

        self.gradW = np.zeros_like(self.W)
        self.gradb = np.zeros_like(self.b)

    def updateOutput(self, input):
        self.output = input @ self.W.T + self.b
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput @ self.W
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        self.gradW = gradOutput.T @ input
        self.gradb = gradOutput.sum(axis=0)

    def zeroGradParameters(self):
        self.gradW.fill(0)
        self.gradb.fill(0)

    def getParameters(self):
        return [self.W, self.b]

    def getGradParameters(self):
        return [self.gradW, self.gradb]

    def __repr__(self):
        s = self.W.shape
        q = 'Linear %d -> %d' %(s[1],s[0])
        return q


## 2. (0.2) SoftMax
- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

$\text{softmax}(x)_i = \frac{\exp x_i} {\sum_j \exp x_j}$

Recall that $\text{softmax}(x) == \text{softmax}(x - \text{const})$. It makes possible to avoid computing exp() from large argument.

In [ ]:
class SoftMax(Module):
    def __init__(self):
         super(SoftMax, self).__init__()

    def updateOutput(self, input):
        self.output = np.subtract(input, input.max(axis=1, keepdims=True))
        self.output = np.exp(self.output)
        self.output = self.output / self.output.sum(axis=1, keepdims=True)
        return self.output

    def updateGradInput(self, input, gradOutput):
        probs = self.output
        self.gradInput = probs * (gradOutput - np.sum(gradOutput * probs, axis=1, keepdims=True))
        return self.gradInput


    def __repr__(self):
        return "SoftMax"

## 3. (0.2) LogSoftMax
- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

$\text{logsoftmax}(x)_i = \log\text{softmax}(x)_i = x_i - \log {\sum_j \exp x_j}$

The main goal of this layer is to be used in computation of log-likelihood loss.

In [ ]:
class LogSoftMax(Module):
    def __init__(self):
         super(LogSoftMax, self).__init__()

    def updateOutput(self, input):
        self.output = np.subtract(input, input.max(axis=1, keepdims=True))
        self.output = self.output - np.log(np.exp(self.output).sum(axis=1, keepdims=True))
        return self.output

    def updateGradInput(self, input, gradOutput):
        probs = np.exp(self.output)
        self.gradInput = gradOutput - probs * gradOutput.sum(axis=1, keepdims=True)
        return self.gradInput


    def __repr__(self):
        return "LogSoftMax"

## 4. (0.3) Batch normalization
One of the most significant recent ideas that impacted NNs a lot is [**Batch normalization**](http://arxiv.org/abs/1502.03167). The idea is simple, yet effective: the features should be whitened ($mean = 0$, $std = 1$) all the way through NN. This improves the convergence for deep models letting it train them for days but not weeks. **You are** to implement the first part of the layer: features normalization. The second part (`ChannelwiseScaling` layer) is implemented below.

- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

The layer should work as follows. While training (`self.training == True`) it transforms input as $$y = \frac{x - \mu}  {\sqrt{\sigma + \epsilon}}$$
where $\mu$ and $\sigma$ - mean and variance of feature values in **batch** and $\epsilon$ is just a small number for numericall stability. Also during training, layer should maintain exponential moving average values for mean and variance:
```
    self.moving_mean = self.moving_mean * alpha + batch_mean * (1 - alpha)
    self.moving_variance = self.moving_variance * alpha + batch_variance * (1 - alpha)
```
During testing (`self.training == False`) the layer normalizes input using moving_mean and moving_variance.

Note that decomposition of batch normalization on normalization itself and channelwise scaling here is just a common **implementation** choice. In general "batch normalization" always assumes normalization + scaling.

In [ ]:
class BatchNormalization(Module):
    EPS = 1e-3
    def __init__(self, alpha = 0.):
        super(BatchNormalization, self).__init__()
        self.alpha = alpha
        self.moving_mean = None
        self.moving_variance = None

    def updateOutput(self, input):
        if self.moving_mean is None:
            self.moving_mean = np.zeros(input.shape[1], dtype=input.dtype)
            self.moving_variance = np.ones(input.shape[1], dtype=input.dtype)

        if self.training:
            self.batch_mean = input.mean(axis=0)
            self.batch_variance = input.var(axis=0)
            self.centered_input = input - self.batch_mean
            self.std_inv = 1.0 / np.sqrt(self.batch_variance + self.EPS)
            self.output = self.centered_input * self.std_inv

            self.moving_mean = self.alpha * self.moving_mean + (1 - self.alpha) * self.batch_mean
            unbiased_var = self.batch_variance * input.shape[0] / max(input.shape[0] - 1, 1)
            self.moving_variance = self.alpha * self.moving_variance + (1 - self.alpha) * unbiased_var
        else:
            self.output = (input - self.moving_mean) / np.sqrt(self.moving_variance + self.EPS)
        return self.output

    def updateGradInput(self, input, gradOutput):
        if not self.training:
            self.gradInput = gradOutput / np.sqrt(self.moving_variance + self.EPS)
            return self.gradInput

        n = input.shape[0]
        x_hat = self.centered_input * self.std_inv
        self.gradInput = (1.0 / n) * self.std_inv * (
            n * gradOutput - gradOutput.sum(axis=0) - x_hat * np.sum(gradOutput * x_hat, axis=0)
        )
        return self.gradInput


    def __repr__(self):
        return "BatchNormalization"

In [ ]:
class ChannelwiseScaling(Module):
    """
       Implements linear transform of input y = \gamma * x + \beta
       where \gamma, \beta - learnable vectors of length x.shape[-1]
    """
    def __init__(self, n_out):
        super(ChannelwiseScaling, self).__init__()

        stdv = 1./np.sqrt(n_out)
        self.gamma = np.random.uniform(-stdv, stdv, size=n_out)
        self.beta = np.random.uniform(-stdv, stdv, size=n_out)

        self.gradGamma = np.zeros_like(self.gamma)
        self.gradBeta = np.zeros_like(self.beta)

    def updateOutput(self, input):
        self.output = input * self.gamma + self.beta
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * self.gamma
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        self.gradBeta = np.sum(gradOutput, axis=0)
        self.gradGamma = np.sum(gradOutput*input, axis=0)

    def zeroGradParameters(self):
        self.gradGamma.fill(0)
        self.gradBeta.fill(0)

    def getParameters(self):
        return [self.gamma, self.beta]

    def getGradParameters(self):
        return [self.gradGamma, self.gradBeta]

    def __repr__(self):
        return "ChannelwiseScaling"

Practical notes. If BatchNormalization is placed after a linear transformation layer (including dense layer, convolutions, channelwise scaling) that implements function like `y = weight * x + bias`, than bias adding become useless and could be omitted since its effect will be discarded while batch mean subtraction. If BatchNormalization (followed by `ChannelwiseScaling`) is placed before a layer that propagates scale (including ReLU, LeakyReLU) followed by any linear transformation layer than parameter `gamma` in `ChannelwiseScaling` could be freezed since it could be absorbed into the linear transformation layer.

## 5. (0.3) Dropout
Implement [**dropout**](https://www.cs.toronto.edu/~hinton/absps/JMLRdropout.pdf). The idea and implementation is really simple: just multimply the input by $Bernoulli(p)$ mask. Here $p$ is probability of an element to be zeroed.

This has proven to be an effective technique for regularization and preventing the co-adaptation of neurons.

While training (`self.training == True`) it should sample a mask on each iteration (for every batch), zero out elements and multiply elements by $1 / (1 - p)$. The latter is needed for keeping mean values of features close to mean values which will be in test mode. When testing this module should implement identity transform i.e. `self.output = input`.

- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

In [ ]:
class Dropout(Module):
    def __init__(self, p=0.5):
        super(Dropout, self).__init__()

        self.p = p
        self.mask = None

    def updateOutput(self, input):
        if self.training:
            self.mask = (np.random.rand(*input.shape) >= self.p).astype(input.dtype) / (1.0 - self.p)
            self.output = input * self.mask
        else:
            self.mask = None
            self.output = input.copy()
        return  self.output

    def updateGradInput(self, input, gradOutput):
        if self.training:
            self.gradInput = gradOutput * self.mask
        else:
            self.gradInput = gradOutput
        return self.gradInput


    def __repr__(self):
        return "Dropout"

#6. (2.0) Conv2d
Implement [**Conv2d**](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html). Use only this list of parameters: (in_channels, out_channels, kernel_size, stride, padding, bias, padding_mode) and fix dilation=1 and groups=1.

In [ ]:
class Conv2d(Module):
    def __init__(self, in_channels, out_channels, kernel_size,
                 stride=1, padding=0, bias=True, padding_mode='zeros'):
        super(Conv2d, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.use_bias = bias
        self.padding_mode = padding_mode

        kh, kw = self._pair(self.kernel_size)
        stdv = 1. / np.sqrt(self.in_channels * kh * kw)

        self.weight = np.random.uniform(
            -stdv, stdv, (self.out_channels, self.in_channels, kh, kw)
        )
        self.gradWeight = np.zeros_like(self.weight)

        if self.use_bias:
            self.bias = np.random.uniform(-stdv, stdv, self.out_channels)
            self.gradBias = np.zeros_like(self.bias)
        else:
            self.bias = None
            self.gradBias = None

    def _pair(self, x):
        return x if isinstance(x, tuple) else (x, x)

    def _same_padding(self, h, w, kh, kw, sh, sw):
        out_h = int(np.ceil(h / sh))
        out_w = int(np.ceil(w / sw))
        pad_h = max((out_h - 1) * sh + kh - h, 0)
        pad_w = max((out_w - 1) * sw + kw - w, 0)
        return (pad_h // 2, pad_h - pad_h // 2, pad_w // 2, pad_w - pad_w // 2)

    def _pad_input(self, x, pads):
        pt, pb, pl, pr = pads
        if self.padding_mode == 'zeros':
            return np.pad(x, ((0, 0), (0, 0), (pt, pb), (pl, pr)), mode='constant')
        raise ValueError(f"Unsupported padding_mode: {self.padding_mode}")

    def _init_params(self):
        if getattr(self, 'gradWeight', None) is None or self.gradWeight.shape != self.weight.shape:
            self.gradWeight = np.zeros_like(self.weight)
        if self.use_bias and (getattr(self, 'gradBias', None) is None or self.gradBias.shape != self.bias.shape):
            self.gradBias = np.zeros_like(self.bias)

    def _get_pads(self, input_shape):
        _, _, h, w = input_shape
        kh, kw = self._pair(self.kernel_size)
        sh, sw = self._pair(self.stride)

        if self.padding == 'same':
            return self._same_padding(h, w, kh, kw, sh, sw)
        if self.padding == 'valid':
            return (0, 0, 0, 0)
        raise ValueError("padding must be 'same' or 'valid'")

    def updateOutput(self, input):
        self._init_params()
        kh, kw = self._pair(self.kernel_size)
        sh, sw = self._pair(self.stride)

        self._pads = self._get_pads(input.shape)
        self.input_padded = self._pad_input(input, self._pads)

        N, C, H_pad, W_pad = self.input_padded.shape
        H_out = (H_pad - kh) // sh + 1
        W_out = (W_pad - kw) // sw + 1

        shape = (N, C, H_out, W_out, kh, kw)
        strides = (
            self.input_padded.strides[0],
            self.input_padded.strides[1],
            sh * self.input_padded.strides[2],
            sw * self.input_padded.strides[3],
            self.input_padded.strides[2],
            self.input_padded.strides[3],
        )
        windows = np.lib.stride_tricks.as_strided(self.input_padded, shape=shape, strides=strides)
        self.cols = windows.transpose(0, 2, 3, 1, 4, 5).reshape(-1, C * kh * kw)

        out = self.cols @ self.weight.reshape(self.out_channels, -1).T
        if self.use_bias:
            out = out + self.bias

        self.output = out.reshape(N, H_out, W_out, self.out_channels).transpose(0, 3, 1, 2)
        return self.output

    def updateGradInput(self, input, gradOutput):
        kh, kw = self._pair(self.kernel_size)
        sh, sw = self._pair(self.stride)
        N, C, H, W = input.shape

        grad2d = gradOutput.transpose(0, 2, 3, 1).reshape(-1, self.out_channels)
        w2d = self.weight.reshape(self.out_channels, -1)
        grad_cols = grad2d @ w2d

        H_out, W_out = gradOutput.shape[2], gradOutput.shape[3]
        grad_windows = grad_cols.reshape(N, H_out, W_out, C, kh, kw).transpose(0, 3, 1, 2, 4, 5)
        grad_padded = np.zeros_like(self.input_padded)

        h_idx = np.arange(H_out)[:, None] * sh + np.arange(kh)[None, :]
        w_idx = np.arange(W_out)[:, None] * sw + np.arange(kw)[None, :]

        np.add.at(
            grad_padded,
            (
                np.arange(N)[:, None, None, None, None, None],
                np.arange(C)[None, :, None, None, None, None],
                h_idx[None, None, :, None, :, None],
                w_idx[None, None, None, :, None, :],
            ),
            grad_windows,
        )

        pt, pb, pl, pr = self._pads
        self.gradInput = grad_padded[:, :, pt:pt + H, pl:pl + W]

        self.gradWeight = (grad2d.T @ self.cols).reshape(self.out_channels, self.in_channels, kh, kw)
        if self.use_bias:
            self.gradBias = grad2d.sum(axis=0)
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        if getattr(self, 'gradWeight', None) is None or self.gradWeight.shape != self.weight.shape:
            self.updateGradInput(input, gradOutput)

    def zeroGradParameters(self):
        if hasattr(self, 'gradWeight'):
            self.gradWeight.fill(0)
        if self.use_bias and hasattr(self, 'gradBias'):
            self.gradBias.fill(0)

    def getParameters(self):
        return [self.weight, self.bias] if self.use_bias else [self.weight]

    def getGradParameters(self):
        return [self.gradWeight, self.gradBias] if self.use_bias else [self.gradWeight]

    def __repr__(self):
        return "Conv2d"


#7. (0.5) Implement [**MaxPool2d**](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) and [**AvgPool2d**](https://pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html). Use only parameters like kernel_size, stride, padding (negative infinity for maxpool and zero for avgpool) and other parameters fixed as in framework.

In [ ]:
class MaxPool2d(Module):
    def __init__(self, kernel_size, stride, padding):
        super(MaxPool2d, self).__init__()

        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

    def _pair(self, x):
        return x if isinstance(x, tuple) else (x, x)

    def updateOutput(self, input):
        kH, kW = self._pair(self.kernel_size)
        sH, sW = self._pair(self.stride)
        pH, pW = self._pair(self.padding)

        self.input_padded = np.pad(
            input,
            ((0, 0), (0, 0), (pH, pH), (pW, pW)),
            mode='constant',
            constant_values=-np.inf,
        )

        N, C, H_pad, W_pad = self.input_padded.shape
        H_out = (H_pad - kH) // sH + 1
        W_out = (W_pad - kW) // sW + 1

        shape = (N, C, H_out, W_out, kH, kW)
        strides = (
            self.input_padded.strides[0],
            self.input_padded.strides[1],
            sH * self.input_padded.strides[2],
            sW * self.input_padded.strides[3],
            self.input_padded.strides[2],
            self.input_padded.strides[3],
        )
        windows = np.lib.stride_tricks.as_strided(self.input_padded, shape=shape, strides=strides)
        flat = windows.reshape(N, C, H_out, W_out, -1)
        self.argmax = flat.argmax(axis=-1)
        self.output = flat.max(axis=-1)
        return  self.output

    def updateGradInput(self, input, gradOutput):
        kH, kW = self._pair(self.kernel_size)
        sH, sW = self._pair(self.stride)
        pH, pW = self._pair(self.padding)

        N, C, H_out, W_out = gradOutput.shape
        grad_padded = np.zeros_like(self.input_padded)
        h_base = np.arange(H_out)[None, None, :, None] * sH
        w_base = np.arange(W_out)[None, None, None, :] * sW
        kh_idx = self.argmax // kW
        kw_idx = self.argmax % kW

        np.add.at(
            grad_padded,
            (
                np.arange(N)[:, None, None, None],
                np.arange(C)[None, :, None, None],
                h_base + kh_idx,
                w_base + kw_idx,
            ),
            gradOutput,
        )

        self.gradInput = grad_padded[:, :, pH:pH + input.shape[2], pW:pW + input.shape[3]]
        return self.gradInput


    def __repr__(self):
        return "MaxPool2d"

class AvgPool2d(Module):
    def __init__(self, kernel_size, stride, padding):
        super(AvgPool2d, self).__init__()

        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding

    def _pair(self, x):
        return x if isinstance(x, tuple) else (x, x)

    def updateOutput(self, input):
        kH, kW = self._pair(self.kernel_size)
        sH, sW = self._pair(self.stride)
        pH, pW = self._pair(self.padding)

        self.input_padded = np.pad(input, ((0, 0), (0, 0), (pH, pH), (pW, pW)), mode='constant')
        N, C, H_pad, W_pad = self.input_padded.shape
        H_out = (H_pad - kH) // sH + 1
        W_out = (W_pad - kW) // sW + 1

        shape = (N, C, H_out, W_out, kH, kW)
        strides = (
            self.input_padded.strides[0],
            self.input_padded.strides[1],
            sH * self.input_padded.strides[2],
            sW * self.input_padded.strides[3],
            self.input_padded.strides[2],
            self.input_padded.strides[3],
        )
        windows = np.lib.stride_tricks.as_strided(self.input_padded, shape=shape, strides=strides)
        self.output = windows.mean(axis=(-1, -2))
        return  self.output

    def updateGradInput(self, input, gradOutput):
        kH, kW = self._pair(self.kernel_size)
        sH, sW = self._pair(self.stride)
        pH, pW = self._pair(self.padding)

        N, C, H_out, W_out = gradOutput.shape
        grad_padded = np.zeros_like(self.input_padded)
        grad_share = gradOutput[:, :, :, :, None, None] / (kH * kW)

        h_idx = np.arange(H_out)[:, None] * sH + np.arange(kH)[None, :]
        w_idx = np.arange(W_out)[:, None] * sW + np.arange(kW)[None, :]

        np.add.at(
            grad_padded,
            (
                np.arange(N)[:, None, None, None, None, None],
                np.arange(C)[None, :, None, None, None, None],
                h_idx[None, None, :, None, :, None],
                w_idx[None, None, None, :, None, :],
            ),
            grad_share,
        )

        self.gradInput = grad_padded[:, :, pH:pH + input.shape[2], pW:pW + input.shape[3]]
        return self.gradInput


    def __repr__(self):
        return "AvgPool2d"

#8. (0.3) Implement **GlobalMaxPool2d** and **GlobalAvgPool2d**. They do not have testing and parameters are up to you but they must aggregate information within channels. Write test functions for these layers on your own.

#9. (0.2) Implement [**Flatten**](https://pytorch.org/docs/stable/generated/torch.flatten.html)

In [ ]:
class Flatten(Module):
    def __init__(self, start_dim=0, end_dim=-1):
        super(Flatten, self).__init__()

        self.start_dim = start_dim
        self.end_dim = end_dim

    def updateOutput(self, input):
        ndim = input.ndim
        start = self.start_dim if self.start_dim >= 0 else self.start_dim + ndim
        end = self.end_dim if self.end_dim >= 0 else self.end_dim + ndim
        new_shape = input.shape[:start] + (-1,) + input.shape[end + 1:]
        self.output = input.reshape(new_shape)
        return  self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput.reshape(input.shape)
        return self.gradInput


    def __repr__(self):
        return "Flatten"

# Activation functions

Here's the complete example for the **Rectified Linear Unit** non-linearity (aka **ReLU**):

In [ ]:
class ReLU(Module):
    def __init__(self):
         super(ReLU, self).__init__()

    def updateOutput(self, input):
        self.output = np.maximum(input, 0)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = np.multiply(gradOutput , input > 0)
        return self.gradInput

    def __repr__(self):
        return "ReLU"

## 10. (0.1) Leaky ReLU
Implement [**Leaky Rectified Linear Unit**](http://en.wikipedia.org/wiki%2FRectifier_%28neural_networks%29%23Leaky_ReLUs). Expriment with slope.

In [ ]:
class LeakyReLU(Module):
    def __init__(self, slope = 0.03):
        super(LeakyReLU, self).__init__()

        self.slope = slope

    def updateOutput(self, input):
        self.output = np.where(input > 0, input, self.slope * input)
        return  self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * np.where(input > 0, 1.0, self.slope)
        return self.gradInput


    def __repr__(self):
        return "LeakyReLU"

## 11. (0.1) ELU
Implement [**Exponential Linear Units**](http://arxiv.org/abs/1511.07289) activations.

In [ ]:
class ELU(Module):
    def __init__(self, alpha = 1.0):
        super(ELU, self).__init__()

        self.alpha = alpha

    def updateOutput(self, input):
        self.output = np.where(input > 0, input, self.alpha * (np.exp(input) - 1))
        return  self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * np.where(input > 0, 1.0, self.alpha * np.exp(input))
        return self.gradInput


    def __repr__(self):
        return "ELU"

## 12. (0.1) SoftPlus
Implement [**SoftPlus**](https://en.wikipedia.org/wiki%2FRectifier_%28neural_networks%29) activations. Look, how they look a lot like ReLU.

In [ ]:
class SoftPlus(Module):
    def __init__(self):
        super(SoftPlus, self).__init__()

    def updateOutput(self, input):
        self.output = np.log1p(np.exp(-np.abs(input))) + np.maximum(input, 0)
        return  self.output

    def updateGradInput(self, input, gradOutput):
        sigmoid = 1.0 / (1.0 + np.exp(-input))
        self.gradInput = gradOutput * sigmoid
        return self.gradInput


    def __repr__(self):
        return "SoftPlus"

#13. (0.2) Gelu
Implement [**Gelu**](https://pytorch.org/docs/stable/generated/torch.nn.GELU.html) activations.

In [ ]:
class Gelu(Module):
    def __init__(self):
        super(Gelu, self).__init__()

    def updateOutput(self, input):
        erf_term = np.vectorize(__import__('math').erf)(input / np.sqrt(2.0))
        self.cdf = 0.5 * (1.0 + erf_term)
        self.output = input * self.cdf
        return  self.output

    def updateGradInput(self, input, gradOutput):
        pdf = np.exp(-0.5 * input ** 2) / np.sqrt(2.0 * np.pi)
        self.gradInput = gradOutput * (self.cdf + input * pdf)
        return self.gradInput


    def __repr__(self):
        return "Gelu"

# Criterions

Criterions are used to score the models answers.

In [ ]:
class Criterion(object):
    def __init__ (self):
        self.output = None
        self.gradInput = None

    def forward(self, input, target):
        """
            Given an input and a target, compute the loss function
            associated to the criterion and return the result.

            For consistency this function should not be overrided,
            all the code goes in `updateOutput`.
        """
        return self.updateOutput(input, target)

    def backward(self, input, target):
        """
            Given an input and a target, compute the gradients of the loss function
            associated to the criterion and return the result.

            For consistency this function should not be overrided,
            all the code goes in `updateGradInput`.
        """
        return self.updateGradInput(input, target)

    def updateOutput(self, input, target):
        """
        Function to override.
        """
        return self.output

    def updateGradInput(self, input, target):
        """
        Function to override.
        """
        return self.gradInput

    def __repr__(self):
        """
        Pretty printing. Should be overrided in every module if you want
        to have readable description.
        """
        return "Criterion"

The **MSECriterion**, which is basic L2 norm usually used for regression, is implemented here for you.
- input:   **`batch_size x n_feats`**
- target: **`batch_size x n_feats`**
- output: **scalar**

In [ ]:
class MSECriterion(Criterion):
    def __init__(self):
        super(MSECriterion, self).__init__()

    def updateOutput(self, input, target):
        self.output = np.sum(np.power(input - target,2)) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        self.gradInput  = (input - target) * 2 / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "MSECriterion"

## 14. (0.2) Negative LogLikelihood criterion (numerically unstable)
You task is to implement the **ClassNLLCriterion**. It should implement [multiclass log loss](http://scikit-learn.org/stable/modules/model_evaluation.html#log-loss). Nevertheless there is a sum over `y` (target) in that formula,
remember that targets are one-hot encoded. This fact simplifies the computations a lot. Note, that criterions are the only places, where you divide by batch size. Also there is a small hack with adding small number to probabilities to avoid computing log(0).
- input:   **`batch_size x n_feats`** - probabilities
- target: **`batch_size x n_feats`** - one-hot representation of ground truth
- output: **scalar**



In [ ]:
class ClassNLLCriterionUnstable(Criterion):
    EPS = 1e-15
    def __init__(self):
        a = super(ClassNLLCriterionUnstable, self)
        super(ClassNLLCriterionUnstable, self).__init__()

    def updateOutput(self, input, target):
        input_clamp = np.clip(input, self.EPS, 1 - self.EPS)
        self.output = -np.sum(target * np.log(input_clamp)) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        input_clamp = np.clip(input, self.EPS, 1 - self.EPS)
        self.gradInput = -target / input_clamp / input.shape[0]
        return self.gradInput


    def __repr__(self):
        return "ClassNLLCriterionUnstable"

## 15. (0.3) Negative LogLikelihood criterion (numerically stable)
- input:   **`batch_size x n_feats`** - log probabilities
- target: **`batch_size x n_feats`** - one-hot representation of ground truth
- output: **scalar**

Task is similar to the previous one, but now the criterion input is the output of log-softmax layer. This decomposition allows us to avoid problems with computation of forward and backward of log().

In [ ]:
class ClassNLLCriterion(Criterion):
    def __init__(self):
        a = super(ClassNLLCriterion, self)
        super(ClassNLLCriterion, self).__init__()

    def updateOutput(self, input, target):
        self.output = -np.sum(target * input) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        self.gradInput = -target / input.shape[0]
        return self.gradInput


    def __repr__(self):
        return "ClassNLLCriterion"

1-я часть задания: реализация слоев, лосей и функций активации - 5 баллов. \\
2-я часть задания: реализация моделей на своих классах. Что должно быть:
  1. Выберите оптимизатор и реализуйте его, чтоб он работал с вами классами. - 1 балл.
  2. Модель для задачи мультирегрессии на выбраных вами данных. Использовать FCNN, dropout, batchnorm, MSE. Пробуйте различные фукнции активации. Для первой модели попробуйте большую, среднюю и маленькую модель. - 1 балл.
  3. Модель для задачи мультиклассификации на MNIST. Использовать свёртки, макспулы, флэттэны, софтмаксы - 1 балла.
  4. Автоэнкодер для выбранных вами данных. Должен быть на свёртках и полносвязных слоях, дропаутах, батчнормах и тд. - 2 балла. \\

Дополнительно в оценке каждой модели будет учитываться:
1. Наличие правильно выбранной метрики и лосс функции.
2. Отрисовка графиков лосей и метрик на трейне-валидации. Проверка качества модели на тесте.
3. Наличие шедулера для lr.
4. Наличие вормапа.
5. Наличие механизма ранней остановки и сохранение лучшей модели.
6. Свитч лося (метрики) и оптимайзера.

# Optimizer

In [ ]:
class Optimizer:
    """
    Базовый класс оптимайзера.
    Умеет работать с:
      1) моделью (у которой есть getParameters / getGradParameters / zeroGradParameters)
      2) или напрямую со списками params / grads.
      то есть с моими классами четко работает
    """
    def __init__(self, model=None, params=None, grads=None):
        self.model = model
        self._params_source = params
        self._grads_source = grads

        if model is None and (params is None or grads is None):
            raise ValueError("Pass either `model` or both `params` and `grads`.")

        self.params = self._current_params()
        self.grads = self._current_grads()

        if len(self.params) != len(self.grads):
            raise ValueError("Number of parameters and gradients must match.")

    def _flatten(self, obj):
        flat = []
        if isinstance(obj, (list, tuple)):
            for item in obj:
                flat.extend(self._flatten(item))
        elif obj is None:
            pass
        else:
            flat.append(obj)
        return flat

    def _current_params(self):
        source = self.model.getParameters() if self.model is not None else self._params_source
        return self._flatten(source)

    def _current_grads(self):
        source = self.model.getGradParameters() if self.model is not None else self._grads_source
        return self._flatten(source)

    def refresh(self):
        self.params = self._current_params()
        self.grads = self._current_grads()

        if len(self.params) != len(self.grads):
            raise ValueError("Number of parameters and gradients must match.")

    def zero_grad(self):
        if self.model is not None and hasattr(self.model, "zeroGradParameters"):
            self.model.zeroGradParameters()
        else:
            for grad in self._current_grads():
                grad.fill(0)

    def step(self):
        raise NotImplementedError

    def set_lr(self, lr):
        self.lr = lr

    def __repr__(self):
        return "Optimizer"


class SGD(Optimizer):
    """
    SGD с опциональным momentum и weight decay.
    """
    def __init__(self, model=None, params=None, grads=None, lr=1e-2, momentum=0.0, weight_decay=0.0):
        super(SGD, self).__init__(model=model, params=params, grads=grads)

        self.lr = lr
        self.momentum = momentum
        self.weight_decay = weight_decay
        self.velocity = [np.zeros_like(param) for param in self.params]

    def step(self):
        self.refresh()

        if len(self.velocity) != len(self.params):
            self.velocity = [np.zeros_like(param) for param in self.params]

        for i, (param, grad) in enumerate(zip(self.params, self.grads)):
            update = grad

            if self.weight_decay != 0.0:
                update = update + self.weight_decay * param

            if self.momentum != 0.0:
                self.velocity[i] = self.momentum * self.velocity[i] - self.lr * update
                param += self.velocity[i]
            else:
                param -= self.lr * update

    def __repr__(self):
        return f"SGD(lr={self.lr}, momentum={self.momentum}, weight_decay={self.weight_decay})"


class AdamW(Optimizer):
    """
    AdamW: Adam + decoupled weight decay.
    """
    def __init__(self, model=None, params=None, grads=None, lr=1e-3,
                 betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-2):
        super(AdamW, self).__init__(model=model, params=params, grads=grads)

        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.weight_decay = weight_decay

        self.m = [np.zeros_like(param) for param in self.params]
        self.v = [np.zeros_like(param) for param in self.params]
        self.t = 0

    def step(self):
        self.refresh()
        self.t += 1

        if len(self.m) != len(self.params):
            self.m = [np.zeros_like(param) for param in self.params]
            self.v = [np.zeros_like(param) for param in self.params]

        for i, (param, grad) in enumerate(zip(self.params, self.grads)):
            if self.weight_decay != 0.0:
                param *= (1.0 - self.lr * self.weight_decay)

            self.m[i] = self.beta1 * self.m[i] + (1.0 - self.beta1) * grad
            self.v[i] = self.beta2 * self.v[i] + (1.0 - self.beta2) * (grad * grad)

            m_hat = self.m[i] / (1.0 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1.0 - self.beta2 ** self.t)

            param -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

    def __repr__(self):
        return (
            f"AdamW(lr={self.lr}, betas=({self.beta1}, {self.beta2}), "
            f"eps={self.eps}, weight_decay={self.weight_decay})"
        )
